# 질병–약물 SVD 추천 – 결과, 한계, 개선

질병×약물 관계 행렬을 Truncated SVD로 근사하고 복원 점수가 높은 약물을 추천했다. 가려 둔 적응증을 상위 10개 추천에서 찾는 Recall@10은 기본 SVD에서 24.31%였다. 행과 열을 정규화한 뒤 SVD를 적용하면 31.63%로 올랐다. 같은 분할에서 팀원의 Strict CBF는 18.26%, PPR은 29.74~30.75%다.

- 방법: 학습 관계를 질병 행 × 약물 열의 0/1 행렬로 만들고, 가장 큰 특이값 k개만 남겨 행렬을 복원한다. 복원 점수가 높은 약물을 추천한다.
- 평가: 팀원 CBF와 같은 규칙으로 적응증을 나눈다(시드 42~46). 검증에서 설정을 고르고, 학습+검증으로 다시 학습해 테스트를 평가한다.
- 기본 결과: rank 10~100 중 100이 가장 좋았고 테스트 Recall@10은 24.31% ± 0.91%p다.
- 한계: 약물이 많은 질병과 많은 질병에 쓰인 약물이 큰 특이값을 지배한다. 관계가 없는 칸을 값 0으로 맞춘다. 학습에 없는 약물은 추천하지 못한다. 유전자 정보를 쓰지 않는다.
- 개선: 행과 열을 그 크기로 나눈 뒤 SVD를 적용하고 열 크기를 다시 곱해 되돌리면 테스트 Recall@10이 24.31%에서 31.63%로, NDCG@10이 0.2332에서 0.3040으로 올랐다. 검증에서도 같은 방향(18.33% → 24.67%)이었다.

## 1. 이 코드는 무엇을 하는가

SVD(특이값 분해)는 행렬 `R`을 `R ≈ Uₖ Σₖ Vₖᵀ`로 근사한다. `k`는 남기는 잠재 차원(rank)이고, 큰 특이값 성분만 남기므로 질병과 약물에 공통으로 나타나는 큰 패턴만 반영된다. 여기서는 `scipy.sparse.linalg.svds`로 가장 큰 특이값 k개를 구한다.

1. 학습에 알려진 질병–약물 관계가 있으면 1, 없으면 0인 희소 행렬 `R`을 만든다.
2. `R`에 Truncated SVD를 적용하고, 복원 행렬에서 대상 질병의 행을 읽어 후보 약물의 점수를 구한다.
3. 대상 질병에 이미 알려진 학습 약물은 제외하고 점수 순으로 정렬한다. 점수가 같으면 약물 ID 순이다.
4. 상위 10개를 추천하고 질병별 Recall@10과 이진 관련성 NDCG@10을 구해 질병 수로 평균한다.

SVD를 적용하는 대상은 질병–약물 행렬 하나다. 질병–유전자 행렬은 만들거나 분해하지 않는다. 유전자 특징을 압축해 비슷한 질병을 찾는 것은 별도의 실험이다. 점수는 확률이나 치료 효과가 아니다.

## 2. 전처리와 분할

1. `indication.csv`의 `disease_id`와 `drug_id`만 쓰고, 같은 쌍이 중복되지 않는지 확인한다. 임상 단계와 이름은 쓰지 않는다.
2. 팀원 CBF 노트북의 규칙(`blake2b(seed|disease_id|drug_id)`)으로 질병별 순서를 만든 뒤 학습·검증·테스트로 나눈다. 시드마다 35,287 / 11,350 / 11,537쌍이고 테스트 질병은 1,662개다.
3. 검증할 때는 학습 관계만으로 행렬을 만들고 검증 관계는 넣지 않는다. 테스트할 때는 검증에서 rank를 고른 뒤 학습+검증으로 행렬을 다시 만든다.
4. 해당 단계의 행렬에 약물이 한 번이라도 나온 경우만 후보로 둔다. 후보에 없는 정답 약물도 Recall의 분모에 남긴다.

## 3. rank별 검증 결과

시드 42~46 다섯 분할의 평균 ± 표본 표준편차다. 검증 NDCG@10 평균이 가장 높은 rank를 골랐고, 같으면 Recall이 높은 쪽, 그다음 작은 rank를 골랐다. 결과를 보고 모델을 골랐기 때문에 이 검증 성능도 낙관적일 수 있다.

In [1]:
validation_results = [{'rank': 10, 'n_splits': 5, 'mean_recall10': 0.13916401237396855, 'sd_recall10': 0.005117309933009692, 'mean_ndcg10': 0.13134334365911518, 'sd_ndcg10': 0.0023598617376023078}, {'rank': 20, 'n_splits': 5, 'mean_recall10': 0.1484538225973858, 'sd_recall10': 0.0029900587621606257, 'mean_ndcg10': 0.14026185678447983, 'sd_ndcg10': 0.0020415909361361888}, {'rank': 50, 'n_splits': 5, 'mean_recall10': 0.16146421564264177, 'sd_recall10': 0.007068135100260783, 'mean_ndcg10': 0.15121869276517189, 'sd_ndcg10': 0.00561528708720354}, {'rank': 100, 'n_splits': 5, 'mean_recall10': 0.18326199023954984, 'sd_recall10': 0.006076466317913028, 'mean_ndcg10': 0.16672368635800364, 'sd_ndcg10': 0.00307539423925969}]
print("rank   검증 Recall@10 평균 ± SD     검증 NDCG@10 평균 ± SD")
for r in validation_results:
    print(f"{r['rank']:>4}     {r['mean_recall10']*100:>6.2f}% ± {r['sd_recall10']*100:.2f}%p        {r['mean_ndcg10']:.4f} ± {r['sd_ndcg10']:.4f}")

rank   검증 Recall@10 평균 ± SD     검증 NDCG@10 평균 ± SD
  10      13.92% ± 0.51%p        0.1313 ± 0.0024
  20      14.85% ± 0.30%p        0.1403 ± 0.0020
  50      16.15% ± 0.71%p        0.1512 ± 0.0056
 100      18.33% ± 0.61%p        0.1667 ± 0.0031


rank가 커질수록 검증 Recall@10은 13.92%에서 18.33%로, NDCG@10은 0.1313에서 0.1667로 계속 올랐다. rank 100이 가장 높다. 이는 시험한 범위에서의 선택이다.

## 4. 팀원 결과와 비교

rank 100으로 각 시드의 학습+검증을 다시 학습하고 해당 시드의 테스트를 평가했다. 시드마다 테스트 질병은 1,662개, 정답은 11,537쌍이다. 검증은 학습 관계만, 테스트는 학습+검증 관계를 쓰므로 3장의 검증 값과 이 표의 테스트 값은 같은 조건이 아니다.

In [1]:
team_comparison = [('SVD (직접 실행, 선택 rank 100)', 24.31371256067519, 0.9140852153080551), ('Strict CBF (팀원 PDF)', 18.2615, 0.1708), ('기본 PPR (팀원 노트북 추가 비교)', 29.74, 0.61), ('조절 PPR (팀원 노트북 추가 비교)', 30.75, 0.55)]
print("방법                                      예비 테스트 Recall@10 평균 ± SD")
for name, value, deviation in team_comparison:
    print(f"{name:<35} {value:>6.2f}% ± {deviation:.2f}%p")

방법                                      예비 테스트 Recall@10 평균 ± SD
SVD (직접 실행, 선택 rank 100)             24.31% ± 0.91%p
Strict CBF (팀원 PDF)                  18.26% ± 0.17%p
기본 PPR (팀원 노트북 추가 비교)                29.74% ± 0.61%p
조절 PPR (팀원 노트북 추가 비교)                30.75% ± 0.55%p


## 5. 한계

- **0의 의미:** 기본 Truncated SVD는 0을 포함한 전체 행렬을 제곱 오차로 근사한다. 0은 아직 관측되지 않았다는 뜻이지 치료 효과가 없다는 증거가 아닌데, 이 방식은 0과 1을 같은 비중으로 맞춘다.
- **큰 행·열의 지배:** 큰 특이값은 약물이 많은 질병과 많은 질병에 쓰인 약물에 크게 좌우된다. 질병당 약물 수의 중앙값은 6개, 약물당 질병 수의 중앙값은 2개이고 최대는 1,776개와 298개로 매우 치우쳐 있다.
- **후보 밖 정답:** 시드 42의 테스트 정답 11,537쌍 중 933쌍(약 8.1%)은 학습+검증에 한 번도 나오지 않은 약물이라 후보 밖이다. 이 정답은 Recall의 분모에 남겼으므로 SVD로는 맞힐 수 없다.
- **정보의 범위:** 질병–약물 관계만 쓴다. 학습 관계가 적은 질병은 행의 정보가 적다.
- **예비 결과:** 정답에 적응증이면서 금기(써서는 안 되는 약으로 기록된 관계)로도 기록된 쌍 339개가 섞여 있고, 분할이 서로 겹치며 이 테스트를 이전에 확인했다. 팀이 정한 정제 데이터와 고정 테스트로 다시 평가해야 한다.

## 6. 개선 실험

기본 SVD의 한계(큰 행·열이 큰 특이값을 지배하는 문제)에 대응해 행과 열을 정규화한 뒤 SVD를 적용했다. 모든 설정은 학습 관계로 만든 행렬로 검증 관계를 평가해 다섯 분할 평균으로 골랐다. 선택한 설정은 학습+검증으로 다시 학습해 테스트를 한 번 평가했다. 추천 규칙(학습 약물 제외, 후보, 지표)은 기본 SVD와 같다. 구현은 `svd_improvement_experiment.py`에 있고, 이 코드가 기본 SVD의 검증 18.33%와 테스트 24.31%를 그대로 재현하는 것을 확인했다. SVD 알고리즘은 그대로이고 입력 행렬과 rank만 바꾼 것이다.

### 6-1. 행·열 크기로 정규화한 뒤 SVD

약물이 많은 질병과 많은 질병에 쓰인 약물이 큰 특이값을 지배하는 문제에 대응한다. 행렬을 `X = Dr^(-b) · R · Dc^(-a)`로 바꾼다. `Dr`은 질병의 학습 약물 수, `Dc`는 약물의 학습 질병 수이고, 각 칸을 `Dr^b`와 `Dc^a`로 나눈다. X에 SVD를 적용해 점수를 구한 뒤, 열에 `Dc^a`를 다시 곱해 원래 크기로 되돌릴 수도 있다. 되돌리지 않으면 드문 약물이 상대적으로 높게 평가된다.

열 지수 a를 0, 0.25, 0.5, 1로, 행 지수 b를 0과 0.5로, 되돌림 여부와 rank 8가지를 조합해 검증했다. 최적값이 격자 안쪽에 있도록 a 0.1~0.3, b 0.5~1.0을 추가로 시험했다. 각 조합에서 가장 좋은 rank의 검증 점수다.

In [1]:
scale = [("기본 (a=0, b=0)", 200, 18.94, 0.1688),
         ("행만 정규화 (b=0.5)", 50, 16.31, 0.1596),
         ("열만 (a=0.25), 되돌림 없음", 200, 17.61, 0.1530),
         ("열만 (a=0.25), 되돌림", 300, 21.52, 0.1820),
         ("열 0.25 + 행 0.5, 되돌림 없음", 200, 22.39, 0.2009),
         ("열 0.25 + 행 0.5, 되돌림", 300, 23.76, 0.2193),
         ("열 0.3 + 행 0.5, 되돌림 (선택)", 300, 24.67, 0.2245)]
print("설정                              rank   검증 Recall@10   검증 NDCG@10")
for name, k, rec, nd in scale:
    print(f"{name:<30} {k:>5}     {rec:>6.2f}%        {nd:.4f}")

설정                              rank   검증 Recall@10   검증 NDCG@10
기본 (a=0, b=0)                    200      18.94%        0.1688
행만 정규화 (b=0.5)                    50      16.31%        0.1596
열만 (a=0.25), 되돌림 없음              200      17.61%        0.1530
열만 (a=0.25), 되돌림                 300      21.52%        0.1820
열 0.25 + 행 0.5, 되돌림 없음           200      22.39%        0.2009
열 0.25 + 행 0.5, 되돌림              300      23.76%        0.2193
열 0.3 + 행 0.5, 되돌림 (선택)          300      24.67%        0.2245


행과 열을 모두 정규화하고 되돌릴 때 가장 높았다. 한쪽만 정규화하면 오히려 떨어지거나 조금만 올랐다. 열 지수가 0.5와 1이면 떨어졌고 행 지수 0.75와 1.0도 0.5보다 낮았다. 선택한 설정은 열 0.3, 행 0.5, 되돌림, rank 300이다.

기본 SVD가 인기 약물 쪽으로 쏠리는지도 확인했다. 시드 42의 검증에서 학습 빈도 상위 50개 약물이 추천 목록에서 차지하는 비율은 기본 SVD 31.4%, 정규화 SVD 27.9%, CBF 32.7%다. 기본 SVD가 특별히 인기 약물에 쏠린 것은 아니다. 따라서 정규화가 좋아진 이유를 인기 약물 억제만으로 설명할 수 없고, 정확한 원인은 확인하지 못했다.

### 6-2. 개선 결과 정리

다섯 분할의 평균 ± 표본 표준편차다.

In [1]:
rows = [("SVD 기본 (rank 100)",                    18.33, 0.1667, 24.31, 0.91, 0.2332),
        ("SVD 정규화 (열 0.3, 행 0.5, rank 300)", 24.67, 0.2245, 31.63, 0.54, 0.3040)]
print("방법                                    검증 Recall  검증 NDCG   테스트 Recall@10 ± SD   테스트 NDCG@10")
for name, vr, vn, tr, sd, tn in rows:
    print(f"{name:<38} {vr:>8.2f}%   {vn:.4f}     {tr:>6.2f}% ± {sd:.2f}%p        {tn:.4f}")

방법                                    검증 Recall  검증 NDCG   테스트 Recall@10 ± SD   테스트 NDCG@10
SVD 기본 (rank 100)                         18.33%   0.1667      24.31% ± 0.91%p        0.2332
SVD 정규화 (열 0.3, 행 0.5, rank 300)          24.67%   0.2245      31.63% ± 0.54%p        0.3040


- 정규화 SVD는 기본 SVD보다 테스트 Recall@10이 7.3%p, NDCG@10이 0.071 높다. 검증에서도 같은 방향이라 테스트에서만 우연히 좋아진 설정은 아니다.
- 같은 분할의 팀원 수치와 비교하면 Strict CBF(18.26%)보다 높고 PPR(29.74~30.75%)과 비슷하거나 약간 높다. PPR 값은 팀원 노트북에서 인용한 것이다. 조정한 항목과 범위가 방법마다 다르고 PPR의 NDCG는 같은 조건으로 제시되지 않았다. 순위를 단정하지 않고 팀 공통 정제 데이터와 고정 테스트에서 다시 비교한다.
- 정규화가 좋아진 이유를 정확히 설명하지는 못한다. 인기 약물 억제만으로는 설명되지 않았다(6-1). 행·열 크기의 치우침(질병당 약물 수 최대 1,776개, 약물당 질병 수 최대 298개)이 큰 특이값을 지배하는 문제를 줄인다고 해석하지만, 이 가설은 직접 검증하지 않았다.
- 정규화에는 열 지수, 행 지수, 되돌림 여부, rank의 네 가지 설정이 있다. 같은 검증 분할에서 고른 값이라 낙관적일 수 있다. 열 지수 0.2~0.3, 행 지수 0.5 주변에서 검증 NDCG가 0.209~0.225로 높게 유지됐지만, 최적값 근처에서도 0.015 정도 차이가 있어 값에 어느 정도 민감하다.
- 한계는 그대로 남는다. 후보 밖 약물은 추천하지 못하고 유전자 정보는 쓰지 않는다.

# 재실행 코드

아래 코드에는 이 노트북의 모든 구현이 들어 있다. 데이터는 포함하지 않는다. `indication.csv`가 있는 경로를 수정하고 `numpy`, `scipy`를 설치한다. 예: `%pip install "numpy>=2,<3" "scipy>=1.15,<2"`. `svds`의 `rng` 인자를 쓰므로 SciPy 1.15 이상이 필요하다. Python 3.12 기준이며 환경에 따라 특이값이 같을 때의 순서나 수치 오차가 달라질 수 있다.

`RUN_EXPERIMENT = False`이면 저장된 결과만 보여 주고 긴 실험은 실행하지 않는다. `True`로 바꾸고 모든 셀을 실행하면 별도 출력 폴더에 결과를 저장한다. 이 코드는 팀원 예비 분할을 재현하기 위한 것이라 적응증이 58,174쌍인지 확인한다. 정제된 팀 공통 데이터에 쓰려면 분할과 후보 정책을 다시 맞춰야 한다.

개선 실험(6장)은 같은 폴더의 `svd_improvement_experiment.py`가 실행한다. 단계는 `diag`(검증 18%와 테스트 24% 진단), `variants`와 `variants2`(rank와 행·열 정규화), `final`(선택한 설정의 검증·테스트 평가)이다. 사용 예는 `python svd_improvement_experiment.py <indication.csv> <disease_gene_t03.csv> <출력 폴더> diag,variants`이며, `final`에는 선택한 설정을 JSON(`{"base_rank":100,"svd":{"col_power":0.3,"row_power":0.5,"rescale":true,"rank":300}}`)으로 다섯 번째 인자에 넘긴다. `final`은 비교용 CBF(IDF, 이웃 50개, 유사도 제곱)도 함께 계산하므로 `disease_gene_t03.csv`가 필요하다.

In [ ]:
from pathlib import Path

# 본인 환경의 경로로 바꾼다.
INDICATION_PATH = Path("optimuskg/processed/indication.csv")
OUTPUT_DIR = Path.home() / "svd_drug_experiment"
RUN_EXPERIMENT = False

print("입력:", INDICATION_PATH)
print("실험 재실행:", RUN_EXPERIMENT)

## 입력 읽기

`indication.csv`를 읽는 함수를 정의한다. 이 셀은 함수만 정의한다.

In [ ]:
"""Reproduce preliminary team splits; tune truncated SVD on validation; report exploratory test.

The five splits overlap one another: pooled validation can include edges in a
different fold's test. Report test results as exploratory, not final holdout.
"""
import argparse
from collections import defaultdict
import csv
import hashlib
import json
from pathlib import Path
from statistics import mean, stdev
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import svds


def read_rows(path):
    with path.open(encoding='utf-8-sig',newline='') as f: return list(csv.DictReader(f))

## 예비 분할 재현

팀원 CBF의 분할 규칙을 재현한다. 질병별로 약물을 해시 순서로 정렬해 테스트, 검증, 학습 순으로 나눈다. 이 셀은 함수만 정의한다.

In [ ]:
def split_pairs(rows,seed):
    grouped=defaultdict(set)
    for r in rows:grouped[r['disease_id']].add(r['drug_id'])
    if sum(map(len,grouped.values()))!=len(rows):raise ValueError('Duplicate indication pairs')
    parts={x:defaultdict(set) for x in ('train','validation','test')}
    for disease,drugs in sorted(grouped.items()):
        def key(drug):
            digest=hashlib.blake2b(f'{seed}|{disease}|{drug}'.encode(),digest_size=8).digest()
            return int.from_bytes(digest,'big'),drug
        ordered=sorted(drugs,key=key);n=len(ordered)
        nt=min(max(1,round(.2*n)),n-2) if n>=3 else 0
        nv=min(max(1,round(.2*n)),n-nt-2) if n-nt>=3 else 0
        for i,drug in enumerate(ordered):
            name='test' if i<nt else 'validation' if i<nt+nv else 'train'
            parts[name][disease].add(drug)
    return {key:dict(value) for key,value in parts.items()}

## 질병–약물 희소 행렬 만들기

학습 관계로 질병 × 약물 이진 희소 행렬을 만든다. 이 셀은 함수만 정의한다.

In [ ]:
def build_matrix(train, diseases):
    disease_ix={d:i for i,d in enumerate(diseases)}
    drugs=sorted({t for values in train.values() for t in values});drug_ix={t:i for i,t in enumerate(drugs)}
    pairs=[(disease_ix[d],drug_ix[t]) for d,values in train.items() for t in values]
    mat=coo_matrix((np.ones(len(pairs),dtype=np.float64),
          ([p[0] for p in pairs],[p[1] for p in pairs])),shape=(len(diseases),len(drugs))).tocsr()
    return mat,drugs,drug_ix,disease_ix

## 후보 순위와 지표 계산

복원 점수로 후보 약물의 순위를 매기고 Recall@10, NDCG@10을 계산한다. 이 셀은 함수만 정의한다.

In [ ]:
def rank_and_evaluate(U,s,Vt,rank,truth,train,drugs,drug_ix,disease_ix,k=10):
    queries=sorted(truth); qidx=[disease_ix[d] for d in queries]
    scores=(U[qidx,:rank]*s[:rank])@Vt[:rank,:]
    recall=[];ndcg=[];predictions=[];details=[];unseen=0;drug_order=np.arange(len(drugs))
    for i,disease in enumerate(queries):
        known=[drug_ix[t] for t in train.get(disease,set()) if t in drug_ix]
        scores[i,known]=-np.inf
        order=np.lexsort((drug_order,-scores[i]))[:k]
        order=order[np.isfinite(scores[i,order])]
        selected=[drugs[int(j)] for j in order];positives=truth[disease]
        hits=[int(t in positives) for t in selected]
        r=sum(hits)/len(positives)
        dcg=sum(h/np.log2(j+2) for j,h in enumerate(hits))
        ideal=sum(1/np.log2(j+2) for j in range(min(k,len(positives))))
        recall.append(r);ndcg.append(dcg/ideal);unseen+=sum(t not in drug_ix for t in positives)
        details.append({'disease_id':disease,'n_relevant':len(positives),'n_recommended':len(selected),
                        'recall10':r,'ndcg10':dcg/ideal})
    return {'evaluated_diseases':len(queries),'relevant_pairs':sum(map(len,truth.values())),
       'unseen_drug_pairs':unseen,'candidate_drugs':len(drugs),'macro_recall10':mean(recall),
       'macro_ndcg10':mean(ndcg),'empty_recommendations':sum(x['n_recommended']==0 for x in details)},details

## SVD와 CSV 저장 함수

`svds`로 Truncated SVD를 구하고 결과 표를 CSV로 저장하는 함수를 정의한다.

In [ ]:
def truncated_svd(matrix, rank):
    if not 0<rank<min(matrix.shape):raise ValueError('Invalid SVD rank')
    U,s,Vt=svds(matrix,k=rank,which='LM',rng=np.random.default_rng(42))
    order=np.argsort(-s,kind='stable')
    return U[:,order],s[order],Vt[order,:]


def write_table(path,rows):
    with path.open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)

## 검증, rank 선택, 테스트 전체 실행

시드마다 검증으로 rank를 평가하고 평균 NDCG@10으로 고른 뒤, 학습+검증으로 다시 학습해 테스트를 평가한다. 이 셀은 함수만 정의한다.

In [ ]:
def run_svd_experiment(indication_path,output,ranks=(10,20,50,100)):
    output.mkdir(parents=True,exist_ok=True)
    rows=read_rows(indication_path)
    if len(rows)!=58174:raise ValueError(f'Expected 58174 unfiltered preliminary pairs; got {len(rows)}')
    diseases=sorted({r['disease_id'] for r in rows});splits={s:split_pairs(rows,s) for s in range(42,47)}
    for seed,parts in splits.items():
        counts=[sum(map(len,parts[n].values())) for n in ('train','validation','test')]
        if counts!=[35287,11350,11537]:raise ValueError(f'Unexpected team split counts {seed}: {counts}')
        if len(parts['test'])!=1662:raise ValueError('Unexpected number of test diseases')
    val_rows=[]
    for seed,parts in splits.items():
        train=parts['train'];truth=parts['validation']
        matrix,drugs,drug_ix,disease_ix=build_matrix(train,diseases)
        U,s,Vt=truncated_svd(matrix,max(ranks))
        for rank in ranks:
            result,_=rank_and_evaluate(U,s,Vt,rank,truth,train,drugs,drug_ix,disease_ix)
            val_rows.append({'seed':seed,'rank':rank,**result})
        print('VALIDATION',seed,[(r['rank'],round(r['macro_ndcg10'],5)) for r in val_rows if r['seed']==seed],flush=True)
    val_summary=[]
    for rank in ranks:
        sub=[r for r in val_rows if r['rank']==rank]
        val_summary.append({'rank':rank,'n_splits':len(sub),'mean_recall10':mean(r['macro_recall10'] for r in sub),
            'sd_recall10':stdev(r['macro_recall10'] for r in sub),
            'mean_ndcg10':mean(r['macro_ndcg10'] for r in sub),'sd_ndcg10':stdev(r['macro_ndcg10'] for r in sub)})
    best=sorted(val_summary,key=lambda r:(-r['mean_ndcg10'],-r['mean_recall10'],r['rank']))[0]
    # The rank is locked before test evaluation. This reuses the CBF preliminary
    # protocol, where pooled validation folds may overlap other folds' tests.
    test_rows=[];test_detail=[]
    for seed,parts in splits.items():
        train={d:parts['train'].get(d,set())|parts['validation'].get(d,set())
               for d in set(parts['train'])|set(parts['validation'])}
        truth=parts['test'];matrix,drugs,drug_ix,disease_ix=build_matrix(train,diseases)
        U,s,Vt=truncated_svd(matrix,best['rank'])
        result,details=rank_and_evaluate(U,s,Vt,best['rank'],truth,train,drugs,drug_ix,disease_ix)
        test_rows.append({'seed':seed,'rank':best['rank'],**result})
        test_detail.extend({'seed':seed,**r} for r in details)
        print('TEST',seed,json.dumps(result),flush=True)
    test_summary={'rank':best['rank'],'n_splits':5,'mean_recall10':mean(r['macro_recall10'] for r in test_rows),
                  'sd_recall10':stdev(r['macro_recall10'] for r in test_rows),
                  'mean_ndcg10':mean(r['macro_ndcg10'] for r in test_rows),
                  'sd_ndcg10':stdev(r['macro_ndcg10'] for r in test_rows)}
    for filename,table in [('validation_by_seed.csv',val_rows),('validation_summary.csv',val_summary),
                           ('test_by_seed.csv',test_rows),('test_per_disease.csv',test_detail)]:write_table(output/filename,table)
    report={'protocol':'CBF preliminary, original 58174 unfiltered positive links, seeds 42-46',
      'zero_policy':'unobserved entries encoded 0 for standard truncated SVD; not proven negatives',
      'test_caution':'Exploratory, 339 contraindication conflicts remain. Across splits, pooled validation overlaps other test folds. Prior tests already observed; not independent final evaluation.',
      'candidate_policy':'all drugs seen in the current train pool; validation or test positives outside candidate pool retained in denominator',
      'ranking_policy':'exclude known training positives, allow finite zero and negative SVD scores, ties by drug id',
      'algorithm_seed':42,'ranks':list(ranks),'selection':'mean validation NDCG@10',
      'selected_rank':best['rank'],'validation':val_summary,'test':test_summary,
      'input_sha256':hashlib.sha256(indication_path.read_bytes()).hexdigest()}
    (output/'summary.json').write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    print('DONE',json.dumps(report,ensure_ascii=False),flush=True)

## 선택적 실행

원본 데이터는 바꾸지 않고 `OUTPUT_DIR`에 요약 CSV와 JSON을 저장한다. 질병–약물 정답 CSV 자체는 출력하지 않는다.

In [ ]:
if RUN_EXPERIMENT:
    run_svd_experiment(INDICATION_PATH, OUTPUT_DIR, ranks=(10,20,50,100))
    print("결과 저장:", OUTPUT_DIR)
else:
    print("재실행하지 않았다. 위에 저장된 결과를 확인한다.")

## 정규화 SVD 개선 실험 코드

아래 함수에 기본 SVD와 별도로 분석한 정규화 실험의 전체 코드를 포함합니다. `diag`, `variants`, `variants2`, `final` 중 실행할 단계를 선택하며, 유전자 파일은 CBF 비교용입니다. 질병–약물 SVD 자체는 유전자 행렬을 사용하지 않습니다. 아래 실행 스위치는 기본적으로 꺼져 있습니다. 여러 rank를 탐색하는 단계는 시간이 오래 걸릴 수 있습니다.


In [ ]:
def run_svd_improvement(indication_path, gene_path, output_dir, stages, final_config=None):
    """SVD improvement experiments on the CBF preliminary splits (seeds 42-46).

    Stages
      diag    : why validation (train only) differs from test (train+validation) at rank 100
      variants: rank sweep and column/row scaling before SVD
      final   : evaluate the chosen SVD setting on validation and test, next to the CBF reference
    Ranks and scaling are chosen on validation only. The test split is evaluated
    once per chosen configuration. The five splits overlap, so test numbers are exploratory.
    """
    import csv, hashlib, json, sys, time
    from collections import defaultdict
    from pathlib import Path
    from statistics import mean, stdev
    import numpy as np
    from scipy.sparse import coo_matrix, csr_matrix

    INDICATION = Path(indication_path)
    GENES = Path(gene_path)
    OUT = Path(output_dir); OUT.mkdir(parents=True, exist_ok=True)
    STAGES = set(stages)
    SEEDS = range(42, 47)
    RANKS = [50, 100, 200, 300, 400, 600, 800, 1200]


    def read_rows(path):
        with path.open(encoding='utf-8-sig', newline='') as f:
            return list(csv.DictReader(f))


    def split_pairs(rows, seed):
        grouped = defaultdict(set)
        for r in rows: grouped[r['disease_id']].add(r['drug_id'])
        parts = {x: defaultdict(set) for x in ('train', 'validation', 'test')}
        for disease, drugs in sorted(grouped.items()):
            def key(drug):
                d = hashlib.blake2b(f'{seed}|{disease}|{drug}'.encode(), digest_size=8).digest()
                return int.from_bytes(d, 'big'), drug
            ordered = sorted(drugs, key=key); n = len(ordered)
            nt = min(max(1, round(.2 * n)), n - 2) if n >= 3 else 0
            nv = min(max(1, round(.2 * n)), n - nt - 2) if n - nt >= 3 else 0
            for i, drug in enumerate(ordered):
                parts['test' if i < nt else 'validation' if i < nt + nv else 'train'][disease].add(drug)
        return {k: dict(v) for k, v in parts.items()}


    def merge(a, b):
        return {d: a.get(d, set()) | b.get(d, set()) for d in set(a) | set(b)}


    def build(train, diseases):
        dix = {d: i for i, d in enumerate(diseases)}
        drugs = sorted({t for v in train.values() for t in v}); tix = {t: i for i, t in enumerate(drugs)}
        pairs = [(dix[d], tix[t]) for d, v in train.items() for t in v]
        R = coo_matrix((np.ones(len(pairs)), ([p[0] for p in pairs], [p[1] for p in pairs])),
                       shape=(len(diseases), len(drugs))).tocsr()
        return R, drugs, tix, dix


    def evaluate(S, truth, train, drugs, tix, dix, k=10, detail=False):
        """S: score rows for sorted(truth) diseases. Known train positives are excluded."""
        queries = sorted(truth); rec = []; ndcg = []; pop_hits = []
        S = S.astype(np.float64, copy=True)
        for i, d in enumerate(queries):
            known = [tix[t] for t in train.get(d, ()) if t in tix]
            S[i, known] = -np.inf
            top = np.argpartition(-S[i], k * 3)[:k * 3]
            top = top[np.lexsort((top, -S[i, top]))][:k]
            top = top[np.isfinite(S[i, top])]
            pos = truth[d]; hits = [int(drugs[j] in pos) for j in top]
            rec.append(sum(hits) / len(pos))
            dcg = sum(h / np.log2(j + 2) for j, h in enumerate(hits))
            ideal = sum(1 / np.log2(j + 2) for j in range(min(k, len(pos))))
            ndcg.append(dcg / ideal)
        return {'recall': mean(rec), 'ndcg': mean(ndcg)}


    def top_ids(S, queries, train, tix, k=10):
        S = S.astype(np.float64, copy=True); out = []
        for i, d in enumerate(queries):
            known = [tix[t] for t in train.get(d, ()) if t in tix]
            S[i, known] = -np.inf
            top = np.argpartition(-S[i], k * 3)[:k * 3]
            out.append(top[np.lexsort((top, -S[i, top]))][:k])
        return out


    class Eig:
        """Rank-k projection of X = Dr^-b R Dc^-a using the Gram matrix eigenvectors."""
        def __init__(self, R, a=0., b=0., kmax=1200):
            dc = np.maximum(np.asarray(R.sum(0)).ravel(), 1.); dr = np.maximum(np.asarray(R.sum(1)).ravel(), 1.)
            self.dc, self.a = dc, a
            X = csr_matrix(R.multiply(dr[:, None] ** -b).multiply(dc[None, :] ** -a))
            G = (X @ X.T).toarray(); w, V = np.linalg.eigh(G)
            self.U = V[:, ::-1][:, :kmax]; self.Z = np.asarray(X.T @ self.U)

        def scores(self, rows, k, rescale=False):
            S = self.U[rows, :k] @ self.Z[:, :k].T
            return S * (self.dc[None, :] ** self.a) if rescale else S



    def idf_similarity(diseases):
        rows = read_rows(GENES); idx = {d: i for i, d in enumerate(diseases)}
        alld = {r['disease_id'] for r in rows}; gd = defaultdict(set)
        for r in rows: gd[r['gene_id']].add(r['disease_id'])
        f = [r for r in rows if r['disease_id'] in idx and float(r['evidence_score']) >= .3]
        genes = sorted({r['gene_id'] for r in f}); gi = {g: i for i, g in enumerate(genes)}
        X = coo_matrix((np.array([float(r['evidence_score']) for r in f]),
                        ([idx[r['disease_id']] for r in f], [gi[r['gene_id']] for r in f])),
                       shape=(len(diseases), len(genes))).tocsr()
        idf = np.array([np.log((len(alld) + 1) / (len(gd[g]) + 1)) + 1 for g in genes])
        X = csr_matrix(X.multiply(idf))
        n = np.sqrt(np.asarray(X.multiply(X).sum(1)).ravel()); inv = np.divide(1., n, out=np.zeros_like(n), where=n > 0)
        from scipy.sparse import diags
        U = diags(inv) @ X; S = (U @ U.T).toarray().astype(np.float32); np.fill_diagonal(S, 0)
        return S


    def cbf_scores(sim, rows, R, k=50, p=2):
        src = np.asarray(R.sum(1)).ravel() > 0; W = np.zeros((len(rows), sim.shape[0]), np.float32)
        for i, r in enumerate(rows):
            s = np.where(src, sim[r], 0); top = np.argpartition(-s, k)[:k]; top = top[s[top] > 0]
            W[i, top] = s[top] ** p
        return np.asarray((R.T @ W.T).T)



    def summarize(rows, keys):
        out = []
        for key in sorted({tuple(r[k] for k in keys) for r in rows}, key=str):
            sub = [r for r in rows if tuple(r[k] for k in keys) == key]
            out.append({**dict(zip(keys, key)), 'n': len(sub),
                        'recall': mean(r['recall'] for r in sub), 'recall_sd': stdev(r['recall'] for r in sub),
                        'ndcg': mean(r['ndcg'] for r in sub), 'ndcg_sd': stdev(r['ndcg'] for r in sub)})
        return out


    def write(name, rows):
        with (OUT / name).open('w', encoding='utf-8', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)


    rows = read_rows(INDICATION); assert len(rows) == 58174
    diseases = sorted({r['disease_id'] for r in rows})
    splits = {s: split_pairs(rows, s) for s in SEEDS}
    t0 = time.time()
    log = lambda *a: print(f'[{time.time() - t0:6.0f}s]', *a, flush=True)

    # ------------------------------------------------------------------ diag
    if 'diag' in STAGES:
        res = []
        for seed, p in splits.items():
            tr, va, te = p['train'], p['validation'], p['test']
            for label, train, truth in [('train->validation', tr, va), ('train->test', tr, te),
                                        ('train+validation->test', merge(tr, va), te)]:
                R, drugs, tix, dix = build(train, diseases); eg = Eig(R, kmax=100)
                q = sorted(truth); S = eg.scores([dix[d] for d in q], 100)
                r = evaluate(S, truth, train, drugs, tix, dix)
                unseen = sum(t not in tix for d in q for t in truth[d])
                res.append({'seed': seed, 'setting': label, 'train_pairs': sum(map(len, train.values())),
                            'truth_pairs': sum(map(len, truth.values())), 'candidates': len(drugs),
                            'unseen_pairs': unseen, **r})
            log('diag seed', seed)
        write('diag_by_seed.csv', res)
        write('diag_summary.csv', summarize(res, ['setting']))
        for r in summarize(res, ['setting']): log(r)

    # ------------------------------------------------------------------ variants
    CONFIGS = [(a, b, rs) for a in (0., .25, .5, 1.) for b in (0., .5) for rs in (False, True) if not (a == 0 and rs)]
    for tag, cfgs, ranks in [('variants', CONFIGS, RANKS),
                             ('variants2', [(a, b, True) for a in (.1, .2, .3) for b in (.5, .75, 1.)], [100, 200, 300, 400, 600])]:
      if tag in STAGES:
        res = []
        for seed, p in splits.items():
            train, truth = p['train'], p['validation']
            R, drugs, tix, dix = build(train, diseases); q = sorted(truth); rws = [dix[d] for d in q]
            for a, b, rs in cfgs:
                eg = Eig(R, a, b, kmax=max(ranks))
                for k in ranks:
                    r = evaluate(eg.scores(rws, k, rs), truth, train, drugs, tix, dix)
                    res.append({'seed': seed, 'col_power': a, 'row_power': b, 'rescale': rs, 'rank': k, **r})
            log(tag, 'seed', seed)
        write(f'{tag}_by_seed.csv', res)
        summ = summarize(res, ['col_power', 'row_power', 'rescale', 'rank'])
        write(f'{tag}_summary.csv', summ)
        for r in sorted(summ, key=lambda r: -r['ndcg'])[:8]: log(r)

    # ------------------------------------------------------------------ final test
    if 'final' in STAGES:
        cfg = final_config
        if cfg is None: raise ValueError('final_config is required for the final stage')
        sim = idf_similarity(diseases); log('similarity ready')
        val, tst = [], []
        for seed, p in splits.items():
            for phase in ('validation', 'test'):
                train = p['train'] if phase == 'validation' else merge(p['train'], p['validation'])
                truth = p[phase]; R, drugs, tix, dix = build(train, diseases)
                q = sorted(truth); rws = [dix[d] for d in q]
                base = Eig(R, kmax=cfg['base_rank']).scores(rws, cfg['base_rank'])
                v = cfg['svd']; best = Eig(R, v['col_power'], v['row_power'], kmax=v['rank']).scores(rws, v['rank'], v['rescale'])
                cb = cbf_scores(sim, rws, R)
                cands = {'svd_baseline': base, 'svd_best': best, 'cbf_idf_k50_p2': cb}
                for name, S in cands.items():
                    r = evaluate(S, truth, train, drugs, tix, dix)
                    (val if phase == 'validation' else tst).append({'seed': seed, 'method': name, **r})
                if phase == 'validation' and seed == 42:
                    # popularity of recommended drugs (share of recommendations among the 50 most common train drugs)
                    freq = np.asarray(R.sum(0)).ravel(); top50 = set(np.argsort(-freq)[:50])
                    diag = {}
                    for name in ('svd_baseline', 'svd_best', 'cbf_idf_k50_p2'):
                        ids = top_ids(cands[name], q, train, tix)
                        diag[name] = float(np.mean([j in top50 for t in ids for j in t]))
                    (OUT / 'popularity_share_seed42.json').write_text(json.dumps(diag, indent=2), encoding='utf-8')
                    log('popularity share', diag)
            log('final seed', seed)
        write('final_validation_by_seed.csv', val); write('final_test_by_seed.csv', tst)
        sv = summarize(val, ['method']); st = summarize(tst, ['method'])
        write('final_validation_summary.csv', sv); write('final_test_summary.csv', st)
        for r in sv: log('VAL', r)
        for r in st: log('TEST', r)


In [ ]:
# 경로를 본인 환경에 맞게 수정한 뒤 필요한 단계만 실행하세요.
RUN_IMPROVEMENT = False
GENE_PATH = Path("optimuskg/processed/disease_gene_t03.csv")
IMPROVEMENT_OUTPUT_DIR = OUTPUT_DIR / "improvement"
IMPROVEMENT_STAGES = ["final"]  # 필요 시 diag, variants, variants2 선택
FINAL_CONFIG = {"base_rank": 100, "svd": {"col_power": 0.3, "row_power": 0.5, "rank": 300, "rescale": True}}
if RUN_IMPROVEMENT:
    run_svd_improvement(INDICATION_PATH, GENE_PATH, IMPROVEMENT_OUTPUT_DIR, IMPROVEMENT_STAGES, FINAL_CONFIG)
else:
    print("정규화 SVD 개선 실험은 재실행하지 않았습니다.")
